In [ ]:
import numpy as np
import pandas as pd
import sklearn
import torch
import scipy.stats

from tqdm import tqdm

from scipy.special import expit, logit
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [ ]:
# PARAMETERS

NUM_RESAMPLES = 300

SEED = 821

SAVE = True
LOAD = False

FONT_SIZE = 14

In [ ]:
rng = np.random.default_rng(seed=SEED)

### Load data (loans)

In [ ]:
# Q_START and Q_END are inclusive

Q_START = "2020Q1"
Q_END   = "2024Q2"

In [ ]:
dfs = []

qstart_i = (int(Q_START[:4]) - 2000) * 4 + int(Q_START[-1]) - 1
qend_i   = (int(Q_END[:4])   - 2000) * 4 + int(Q_END[-1])   - 1

for q in tqdm(range(qstart_i, qend_i+1)):
    df_in = pd.read_csv(f'./data/FannieMae_SingleFamilyLoan/{2000 + q//4}Q{1+(q%4)}_stat.csv', low_memory=False)
    df_filt = df_in.loc[:,'NET_LOSS'].isna()
    df_filt.loc[:] = True
    df_filt.loc[::] = False
    dfs.append(df_in[~df_filt])
  
df_all = pd.concat(dfs, ignore_index=True)

In [ ]:
print(df_all.columns)
df_all.loc[:,'F30_DTE']

In [ ]:
df_small = df_all.loc[:,['orig_rt', 'orig_amt', 'orig_trm', # FEATURES
                         'oltv', 'ocltv', 'num_bo', 'dti',
                         'CSCORE_B', 'FTHB_FLG', 'purpose',
                         'PROP_TYP', 'NUM_UNIT', 'occ_stat',
                         'CSCORE_C','ORIG_VAL', 'relo_flg',
                         'mi_pct', 'MI_TYPE',
#
                         'F30_DTE',                        # TARGET
#
                         'state',                           # FEATURE OR DEMOGRAPHIC
#
                         'zip_3', 'msa']]                  # DEMOGRAPHIC

df_small

In [ ]:
cat_cols = ['FTHB_FLG','purpose','PROP_TYP','occ_stat','relo_flg', 'MI_TYPE', 'state']

df_processed = df_small.copy()
for c in cat_cols:
    df_processed.loc[:,c] = df_processed.loc[:,c].astype('category').cat.codes

df_processed

### Final data processing

In [ ]:
df_features = df.drop(columns=['zip_3','msa','F30_DTE'])
df_dem = df[['state','zip_3','msa']]
sr_labels = df['F30_DTE'].isna()

In [ ]:
# processing here to filter dataset if desired

In [ ]:
df_features_train, df_features_test, \
sr_labels_train, sr_labels_test, \
df_dem_train, df_dem_test = train_test_split(df_features,sr_labels, df_dem, random_state=SEED)

### Initial RF on data

In [ ]:
if LOAD:
    train_prob = np.load('saved_data/train_prob.npy')
    test_prob  = np.load('saved_data/test_prob.npy')

else:
    HGB_params = {
    #    'min_samples_leaf': 100,
        'max_iter': 1000,
    #    'learning_rate': 1,
    #    'max_depth': 5
    #    'early_stopping': False,
        'categorical_features': [i for c1 in cat_cols for (i, c2) in enumerate(df_features_train.index) if c1==c2],
    #    'class_weight': 'balanced',
    }
    
    HGB = HistGradientBoostingClassifier(**HGB_params)
    
    HGB.fit(df_features_train, sr_labels_train)
    
    train_prob = HGB.predict_proba(df_features_train)[:,1]
    test_prob = HGB.predict_proba(df_features_test)[:,1]

In [ ]:
print(f'Train accuracy: {np.sum((train_prob > 0.5) == sr_labels_train) / sr_labels_train.shape[0]}')
print(f'Test accuracy: { np.sum((test_prob > 0.5)  == sr_labels_test) /  sr_labels_test.shape[0]}')
print()

loss_train = -np.sum(   sr_labels_train  * np.log(    train_prob)
                    +(1-sr_labels_train) * np.log(1 - train_prob))\
              / sr_labels_train.shape[0]
loss_test =  -np.sum(   sr_labels_test   * np.log(    test_prob)
                    +(1-sr_labels_test)  * np.log(1 - test_prob)) \
              / sr_labels_test.shape[0]

print(f'Train loss (avg): {loss_train}')
print(f'Test loss (avg): { loss_test}')

print()
print()

pos_prob = np.sum(sr_labels_train) /sr_labels_train.shape[0]
loss_naive = -np.sum(   sr_labels_train  * np.log(    pos_prob)
                    +(1-sr_labels_train) * np.log(1 - pos_prob))\
              / sr_labels_train.shape[0]

print(f'Naive accuracy: {np.sum(sr_labels_train) / sr_labels_train.shape[0]}')
print(f'Naive loss (avg): {loss_naive}')

In [ ]:
confusion_matrix = \
  [[np.sum(((train_prob > 0.5) == p_val) & (sr_labels_train == t_val)) for p_val in [0,1]] for t_val in [0,1]]
plt.matshow(confusion_matrix)
confusion_matrix

### Compute regret

In [ ]:
if LOAD:
    resamp_from_pred_probs_train = np.load('saved_data/resamp_from_pred_probs_train.npy')
    resamp_from_pred_probs_test  = np.load('saved_data/resamp_from_pred_probs_test.npy')

else:
    
    resamp_from_pred_probs_train = np.zeros((df_features_train.shape[0], NUM_RESAMPLES))
    resamp_from_pred_probs_test  = np.zeros((df_features_test.shape[0], NUM_RESAMPLES))
    
    for i in tqdm(range(NUM_RESAMPLES)):
                
            HGB_from_pred = HistGradientBoostingClassifier(**HGB_params)
            HGB_from_pred.fit(df_features_train, rng.binomial(1, train_prob))
    
            resamp_from_pred_probs_train[:,i] = HGB_from_pred.predict_proba(df_features_train)[:,1]
            resamp_from_pred_probs_test[:,i]  = HGB_from_pred.predict_proba(df_features_test)[:,1]

In [ ]:
if SAVE:
    np.save('saved_data/resamp_from_pred_probs_train.npy', resamp_from_pred_probs_train)
    np.save('saved_data/resamp_from_pred_probs_test.npy', resamp_from_pred_probs_test)
    np.save('saved_data/train_prob.npy', train_prob)
    np.save('saved_data/test_prob.npy',  test_prob)

In [ ]:
regret_pred_probs_train = np.var(resamp_from_pred_probs_train, axis=1, ddof=1)
regret_pred_probs_test  = np.var(resamp_from_pred_probs_test, axis=1, ddof=1)

In [ ]:
(regret_pred_probs_train < 0.0001).sum() / regret_pred_probs_train.shape[0]

In [ ]:
(regret_pred_probs_train < 0.0001).sum()

In [ ]:
np.max(regret_pred_probs_train)

### Regret

In [ ]:
plt.rcParams.update({'font.size': FONT_SIZE})

plt.hist([regret_pred_probs_train],
         bins=200,
         density=True,
         log=True)
plt.xlabel('Regret')
plt.plot(np.linspace(0,0.025,100), 3.5e-10/np.linspace(0.00001,0.025,100)**3, '--', color='black')
plt.legend(['$y=\\frac{3.5\\cdot 10^{-10}}{x^3}$', 'Density'])
plt.xlim([0,0.025])
plt.ylim([10**-5,10**4])
plt.savefig('plots/paper/fannie-mae-hist-train-log.pdf', bbox_inches='tight')
plt.show()

plt.hist([regret_pred_probs_train],
         bins=10**np.linspace(np.log10(regret_pred_probs_train.min()),
                              np.log10(regret_pred_probs_train.max()),
                              200),
         density=True,
         log=True)
plt.xlabel('Regret')
plt.xscale('log')
plt.plot(10**np.linspace(-7,-1,100),  3.5e-10/10**(np.linspace(-7,-1,100)*3), '--', color='black')
plt.xlim([10**-7,10**-1.5])
plt.ylim([10**-5,10**5.5])
plt.legend(['$y=\\frac{3.5\\cdot 10^{-10}}{x^3}$', 'Density'])
plt.savefig('plots/paper/fannie-mae-hist-train-loglog.pdf', bbox_inches='tight')
plt.show()

In [ ]:
plt.rcParams.update({'font.size': FONT_SIZE})

plt.hist([regret_pred_probs_test],
         bins=200,
         density=True,
         log=True)
plt.xlabel('Regret')
plt.plot(np.linspace(0,0.025,100), 3.5e-10/np.linspace(0.00001,0.025,100)**3, '--', color='black')
plt.legend(['$y=\\frac{3.5\\cdot 10^{-10}}{x^3}$', 'Density'])
plt.xlim([0,0.025])
plt.ylim([10**-5,10**4])
plt.savefig('plots/paper/fannie-mae-hist-test-log.pdf', bbox_inches='tight')
plt.show()

plt.hist([regret_pred_probs_test],
         bins=10**np.linspace(np.log10(regret_pred_probs_test.min()),
                              np.log10(regret_pred_probs_test.max()),
                              200),
         density=True,
         log=True)
plt.xlabel('Regret')
plt.xscale('log')
plt.plot(10**np.linspace(-7,-1,100),  3.5e-10/10**(np.linspace(-7,-1,100)*3), '--', color='black')
plt.xlim([10**-7,10**-1.5])
plt.ylim([10**-5,10**5.5])
plt.legend(['$y=\\frac{3.5\\cdot 10^{-10}}{x^3}$', 'Density'])
plt.savefig('plots/paper/fannie-mae-hist-test-loglog.pdf', bbox_inches='tight')
plt.show()

In [ ]:
plt.rcParams.update({'font.size': FONT_SIZE})

plt.hist([regret_pred_probs_train[sr_labels_train == c] for c in [0]],
         log=True,
         bins=np.linspace(0,regret_pred_probs_train.max(), 50),
         density=True,
         alpha=0.5)
plt.hist([regret_pred_probs_train[sr_labels_train == c] for c in [1]],
         log=True,
         bins=np.linspace(0,regret_pred_probs_train.max(), 50),
         density=True,
         alpha=0.5)
plt.xlim([0,regret_pred_probs_train.max()])
plt.legend(["30 day deliquency exists", "No deliquency"])
plt.ylabel('Density')
plt.xlabel('Regret')
plt.savefig('plots/paper/fannie-mae-hist-train-label.pdf', bbox_inches='tight')
plt.show()

In [ ]:
plt.rcParams.update({'font.size': FONT_SIZE})

plt.hist([regret_pred_probs_test[sr_labels_test == c] for c in [0]],
         log=True,
         bins=np.linspace(0,regret_pred_probs_test.max(), 50),
         density=True,
         alpha=0.5)
plt.hist([regret_pred_probs_test[sr_labels_test == c] for c in [1]],
         log=True,
         bins=np.linspace(0,regret_pred_probs_test.max(), 50),
         density=True,
         alpha=0.5)
plt.xlim([0,regret_pred_probs_test.max()])
plt.legend(["30 day deliquency exists", "No deliquency"])
plt.ylabel('Density')
plt.xlabel('Regret')
plt.savefig('plots/paper/fannie-mae-hist-test-label.pdf', bbox_inches='tight')
plt.show()